# Antigen Design Workflow

Edit the `query` value below to a gene symbol, UniProt entry name, or accession, then run the notebook top to bottom. The analysis writes the JSON report, Markdown report, and image assets into `outputs/`.

In [ ]:
import importlib
import inspect
import json
import os
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src' / 'agdesign2').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError(
        f'Could not find the project root from {cwd}. Start Jupyter inside the AgDesign2 repo or open the notebook from there.'
    )


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.agdesign2' / 'mplconfig'))
os.environ.setdefault('XDG_CACHE_HOME', str(ROOT / '.agdesign2' / 'cache_home'))
os.environ['PATH'] = os.pathsep.join([
    '/opt/homebrew/bin',
    '/usr/local/bin',
    os.environ.get('PATH', ''),
])

for module_name in list(sys.modules):
    if module_name == 'agdesign2' or module_name.startswith('agdesign2.'):
        del sys.modules[module_name]

from agdesign2 import AnalysisConfig
from agdesign2.pipeline import analyze_target

print(f'Project root: {ROOT}')
print(f'agdesign2 package: {Path(sys.modules["agdesign2"].__file__).resolve()}')
print(f'analyze_target signature: {inspect.signature(analyze_target)}')


In [ ]:
query = 'ACVR1C'  # Gene symbol, UniProt entry name, or accession.
output_dir = ROOT / 'outputs'

config = AnalysisConfig(
    generate_assets=True,
    render_structure_images=True,
    render_quality_plots=True,
    verbose_progress=True,
    enable_complex_portal=False,
)

report = analyze_target(query, config=config, output_dir=output_dir)
report_stem = f"{report.target.entry_name.lower()}_report"
report_json = output_dir / f"{report_stem}.json"
report_md = output_dir / f"{report_stem}.md"
assets_dir = output_dir / f"{report_stem}_assets"
asset_file_count = len(list(assets_dir.glob('*.png'))) if assets_dir.exists() else 0
report_asset_refs = sum(1 for detail in report.construct_details if detail.structure_image or detail.quality_plot)

run_summary = {
    'input_query': query,
    'resolved_target': report.target.entry_name,
    'gene_symbol': report.target.gene_symbol,
    'json_report': str(report_json),
    'markdown_report': str(report_md),
    'assets_dir': str(assets_dir),
    'construct_count': len(report.construct_recommendations),
    'asset_files_found': asset_file_count,
    'constructs_with_asset_refs': report_asset_refs,
}
print(json.dumps(run_summary, indent=2))


In [ ]:
summary = {
    'target': report.target.to_dict(),
    'ectodomain': report.ectodomain.to_dict() if report.ectodomain else None,
    'constructs': [item.to_dict() for item in report.construct_recommendations[:10]],
    'homology': [item.to_dict() for item in report.ectodomain_homology],
    'cross_reactivity_hits': [item.to_dict() for item in report.cross_reactivity_hits[:10]],
    'notes': [item.to_dict() for item in report.notes],
}
display(Markdown('## Analysis Summary'))
print(json.dumps(summary, indent=2))


In [ ]:
display(Markdown('## Detailed Construct Preview'))

def safe_filename(text: str) -> str:
    return ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in text)

def resolve_asset_path(relative_path: str | None, fallback_name: str) -> Path | None:
    candidates = []
    if relative_path:
        candidates.append(output_dir / relative_path)
        candidates.append(assets_dir / Path(relative_path).name)
    candidates.append(assets_dir / fallback_name)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

max_constructs_to_preview = 6
previewed = 0
for detail in report.construct_details:
    safe_name = safe_filename(detail.name)
    structure_path = resolve_asset_path(detail.structure_image, f'{safe_name}_structure.png')
    quality_path = resolve_asset_path(detail.quality_plot, f'{safe_name}_quality.png')
    evidence = '; '.join(detail.evidence) if detail.evidence else 'None'
    warnings = '; '.join(detail.warnings) if detail.warnings else 'None'
    homolog_lines = []
    for homolog in detail.homologs:
        if homolog.available and homolog.start is not None and homolog.end is not None:
            identity = f"; identity to human {homolog.identity_to_human:.2f}%" if homolog.identity_to_human is not None else ''
            homolog_lines.append(f"- {homolog.species}: `{homolog.entry_name or homolog.accession}` {homolog.start}-{homolog.end}{identity}")
        else:
            homolog_lines.append(f"- {homolog.species}: unavailable")
    homolog_text = '\n'.join(homolog_lines) if homolog_lines else '- none'
    sequence_preview = detail.sequence if len(detail.sequence) <= 140 else detail.sequence[:140] + '...'
    construct_md = f"""
### {detail.name}

- Human: `{detail.human_entry_name or report.target.entry_name}` {detail.start}-{detail.end}
- Length: {detail.length} aa
- Classification: `{detail.classification or 'n/a'}`
- Score: {detail.score:.1f}
- Rationale: {detail.rationale}
- Domain summary: {detail.domain_summary or 'None'}
- Evidence: {evidence}
- Warnings: {warnings}
- Sequence: `{sequence_preview}`

Homologs:
{homolog_text}
"""
    display(Markdown(construct_md))
    if structure_path:
        display(Image(filename=str(structure_path), width=320))
    if quality_path:
        display(Image(filename=str(quality_path), width=420))
    previewed += 1
    if previewed >= max_constructs_to_preview:
        break

if previewed == 0:
    print(f'No construct images were found in {assets_dir}.')


In [ ]:
import matplotlib.pyplot as plt

if report.ectodomain:
    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.hlines(1, report.ectodomain.start, report.ectodomain.end, linewidth=8, color='#1f77b4', label='Ectodomain')
    for region in report.structural_regions:
        ax.hlines(0.8, region.start, region.end, linewidth=6, color='#ff7f0e', alpha=0.8)
    for construct in report.construct_recommendations[:5]:
        ax.hlines(0.55, construct.start, construct.end, linewidth=4, alpha=0.7)
    ax.set_title(f"{report.target.entry_name} ectodomain and top constructs")
    ax.set_xlabel('Residue')
    ax.set_yticks([])
    display(fig)
    plt.close(fig)
else:
    print('No ectodomain available for plotting.')


In [ ]:
display(Markdown('## Markdown Report Preview'))
if report_md.exists():
    display(Markdown(report_md.read_text(encoding='utf-8')))
else:
    print(f'Markdown report not found yet: {report_md}')
